In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import numpy as np
import pandas as pd
import networkx as nx
import warnings
warnings.filterwarnings("ignore")
# 设置样式（避免中文乱码）
plt.rcParams['font.family'] = ['DejaVu Sans', 'Arial', 'sans-serif']
sns.set_style("whitegrid")
import pandas as pd
import numpy as np
import networkx as nx
from collections import Counter
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import confusion_matrix, roc_auc_score, f1_score, precision_score, recall_score, classification_report
import warnings
import time

warnings.filterwarnings("ignore")
import torch
import numpy as np
import pandas as pd
from torch_geometric.data import Data
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from torch_geometric.nn import global_mean_pool
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, recall_score, precision_score, confusion_matrix
import torch.optim as optim
import torch.nn as nn
from torch_geometric.nn import global_mean_pool

In [2]:
# 请将路径替换为您的实际文件路径
FILE_PATH = 'E:\\ponzi\\data\\2PonziContract_20220701.csv'
ADDRESS_COLUMN = 'address'
LABEL_COLUMN = 'label'

def clean_and_analyze_data(df: pd.DataFrame, addr_col: str, label_col: str) -> pd.DataFrame:
    """
    分析并清理 DataFrame，剔除标签不一致和重复的地址记录。
    
    Args:
        df: 原始数据 DataFrame.
        addr_col: 地址列名.
        label_col: 标签列名.

    Returns:
        pd.DataFrame: 清理后的 DataFrame，保证地址唯一且标签明确。
    """
    original_len = len(df)
    
    # --- 1. 识别标签不一致的地址 (核心清理步骤) ---
    
    # 统计每个地址有多少个不同的标签值
    label_counts = df.groupby(addr_col)[label_col].nunique()
    inconsistent_addresses = label_counts[label_counts > 1].index.tolist()
    
    # 移除所有记录 associated with inconsistent addresses
    df_step1 = df[~df[addr_col].isin(inconsistent_addresses)].copy()
    
    removed_inconsistent_count = len(inconsistent_addresses)
    removed_inconsistent_rows = original_len - len(df_step1)

    print("\n--- 数据清理报告 ---")
    print(f"原始数据行数: {original_len}")
    
    if removed_inconsistent_count > 0:
        print(f"❌ 1. 移除 '标签不一致' 的地址行数: {removed_inconsistent_rows} (涉及 {removed_inconsistent_count} 个地址)")
        print(f"   这些地址被完全移除以避免标签歧义。")
    else:
        print("✅ 1. 未发现标签不一致的地址，无需移除。")

    # --- 2. 移除一致标签的重复记录 ---
    
    # 对于剩余的地址 (其标签已保证一致)，移除重复行，只保留第一条
    df_cleaned = df_step1.drop_duplicates(subset=[addr_col], keep='first').copy()
    
    removed_consistent_duplicate_rows = len(df_step1) - len(df_cleaned)
    
    if removed_consistent_duplicate_rows > 0:
        print(f"♻️ 2. 移除 '标签一致但重复' 的地址行数: {removed_consistent_duplicate_rows}")
    else:
        print("✅ 2. 未发现标签一致的重复记录。")
        
    print(f"--------------------------------------------------")
    print(f"清理后的最终行数: {len(df_cleaned)}")
    print(f"最终唯一地址数: {len(df_cleaned[addr_col].unique())}")
    
    return df_cleaned


def analyze_contract_data(file_path: str, addr_col: str, label_col: str):
    """
    主分析函数，读取文件并进行清理。
    """
    try:
        # 1. 读取数据
        df = pd.read_csv(file_path)
        print(f"成功读取文件：{file_path}")
    except FileNotFoundError:
        print(f"错误：文件未找到，请检查路径：{file_path}")
        return
    except Exception as e:
        print(f"读取文件时发生错误: {e}")
        return

    # 2. 清理和分析数据
    cleaned_df = clean_and_analyze_data(df, addr_col, label_col)
    
    return cleaned_df

In [3]:
cleaned_data = analyze_contract_data(FILE_PATH, ADDRESS_COLUMN, LABEL_COLUMN)
ponzi_contract = cleaned_data

成功读取文件：E:\ponzi\data\2PonziContract_20220701.csv

--- 数据清理报告 ---
原始数据行数: 6498
❌ 1. 移除 '标签不一致' 的地址行数: 30 (涉及 15 个地址)
   这些地址被完全移除以避免标签歧义。
✅ 2. 未发现标签一致的重复记录。
--------------------------------------------------
清理后的最终行数: 6468
最终唯一地址数: 6468


In [4]:
# 定义操作码类别 (保持不变)
opcode_categories = {
    "Stack Operations": ["POP", "PUSH1", "PUSH2", "PUSH3", "PUSH4", "PUSH5", "PUSH6", "PUSH7", "PUSH8", "PUSH9", "PUSH10", "PUSH11", "PUSH12", "PUSH13", "PUSH14", "PUSH15", "PUSH16", "PUSH17", "PUSH18", "PUSH19", "PUSH20", "PUSH21", "PUSH22", "PUSH23", "PUSH24", "PUSH25", "PUSH26", "PUSH27", "PUSH28", "PUSH29", "PUSH30", "PUSH31", "PUSH32", "DUP1", "DUP2", "DUP3", "DUP4", "DUP5", "DUP6", "DUP7", "DUP8", "DUP9", "DUP10", "DUP11", "DUP12", "DUP13", "DUP14", "DUP15", "DUP16", "SWAP1", "SWAP2", "SWAP3", "SWAP4", "SWAP5", "SWAP6", "SWAP7", "SWAP8", "SWAP9", "SWAP10", "SWAP11", "SWAP12", "SWAP13", "SWAP14", "SWAP15", "SWAP16"],
    "Data Storage and Memory Operations": ["SLOAD", "SSTORE", "MLOAD", "MSTORE", "MSTORE8", "CODECOPY", "EXTCODECOPY"],
    "Mathematics and Logical Operations": ['ADD', 'MUL', 'SUB', 'DIV', 'SDIV', 'MOD', 'SMOD', 'EXP', 'SIGNEXTEND', 'LT', 'GT', 'SLT', 'SGT', 'EQ', 'AND', 'OR', 'XOR', 'NOT', 'BYTE', 'SHL', 'SHR', 'SAR', 'ISZERO'],
    "Control Flow Operations": ["JUMP", "JUMPI", "JUMPDEST", "STOP", "RETURN", "REVERT", "SELFDESTRUCT", "PC"],
    "Contract Call and Message Operations": ["CALL", "CALLCODE", "DELEGATECALL", "STATICCALL", "CREATE", "CREATE2", "SELFDESTRUCT", "RETURN", "REVERT", "STOP"],
    "Environment and State Information": ["GAS", "GASPRICE", "GASLIMIT", "TIMESTAMP", "NUMBER", "DIFFICULTY", "BASEFEE", "COINBASE", "BLOCKHASH", "ADDRESS", "BALANCE", "ORIGIN", "CALLER", "CALLVALUE", "CALLDATALOAD", "CALLDATASIZE", "CALLDATACOPY", "CODESIZE", "CODECOPY", "EXTCODESIZE", "EXTCODECOPY", "EXTCODEHASH", "MSIZE", "RETURNDATASIZE", "RETURNDATACOPY", "PC"],
    "Data Processing and Hashing": ["SHA3", "CALLDATALOAD", "CALLDATASIZE", "CALLDATACOPY", "RETURNDATASIZE", "RETURNDATACOPY", "CODECOPY", "EXTCODECOPY"],
    "Events and Logs Operations": ["LOG0", "LOG1", "LOG2", "LOG3", "LOG4"],
    "Exception and Termination": ["STOP", "RETURN", "REVERT", "INVALID", "SELFDESTRUCT"]
}


In [ ]:
def construct_EthTrace_graph(ponzi_contract, addr, opcode_categories):
    opcode = ponzi_contract[ponzi_contract['address'] == addr]['opcode'].values[0]
    opcode_lines = opcode.split('\n')
    opcode_sequence_str = opcode_lines[0]
    opcode_list = [op for op in opcode_sequence_str.split(' ') if op]
    
    trace_data = []
    for i in range(len(opcode_list) - 1):
        from_op = opcode_list[i]
        to_op = opcode_list[i + 1]
        position = i + 1
        trace_data.append((from_op, to_op, position))
    
    TraceChain_df = pd.DataFrame(trace_data, columns=['from_opcode_id', 'to_opcode_id', 'position'])
    
    # one-hot 编码
    all_opcodes = pd.unique(TraceChain_df[['from_opcode_id', 'to_opcode_id']].values.ravel('K'))
    for opcode in all_opcodes:
        col_name = f'opcode_{opcode}'
        TraceChain_df[col_name] = ((TraceChain_df['from_opcode_id'] == opcode) | 
                                  (TraceChain_df['to_opcode_id'] == opcode)).astype(int)
    
    # 地址编码
    unique_addresses = pd.unique(TraceChain_df[['from_opcode_id', 'to_opcode_id']].values.ravel('K'))
    address_encoding = {addr: idx for idx, addr in enumerate(unique_addresses)}
    
    TraceChain_df['from'] = TraceChain_df['from_opcode_id'].map(address_encoding)
    TraceChain_df['to'] = TraceChain_df['to_opcode_id'].map(address_encoding)
    TraceChain_df = TraceChain_df.drop(columns=['from_opcode_id', 'to_opcode_id'])
    
    # ====== 超边构建：一个类别 = 一个超边 ======
    opcodes = opcode_sequence_str.split()
    nodes = list(set(opcodes))
    
    categorized = {category: [] for category in opcode_categories.keys()}
    categorized['Other'] = []
    
    for op in nodes:
        found = False
        for category, op_list in opcode_categories.items():
            if op in op_list:
                encoded_op = address_encoding.get(op, op)
                categorized[category].append(encoded_op)
                found = True
                break
        if not found:
            encoded_op = address_encoding.get(op, op)
            categorized['Other'].append(encoded_op)
    
    # 包装成 list of list
    hyperedge_dict = {}
    for category, node_list in categorized.items():
        if node_list:
            hyperedge_dict[category] = [node_list]  # ← 一个超边
    
    return TraceChain_df, hyperedge_dict

In [ ]:
def extract_transactional_features(df):
    """提取账户的交易活跃度特征，并合并 df 中除 from/to/position 外的交易级特征（按 from 聚合）。"""
    
    # ------------------- 1. 原有活跃度特征 -------------------
    # 获取所有唯一账户 ID
    all_nodes = pd.Series(df['from'].tolist() + df['to'].tolist()).unique()
    
    # 1.1 出度：账户发起交易数量
    out_degree = df['from'].value_counts().rename('out_degree')
    
    # 1.2 入度：账户接收交易数量
    in_degree = df['to'].value_counts().rename('in_degree')
    
    # 1.3 基础特征 DataFrame
    features_df = pd.DataFrame(index=all_nodes)
    features_df = features_df.join(out_degree, how='left').fillna(0)
    features_df = features_df.join(in_degree, how='left').fillna(0)
    
    # 1.4 衍生特征
    features_df['total_tx'] = features_df['out_degree'] + features_df['in_degree']
    features_df['net_flow'] = features_df['out_degree'] - features_df['in_degree']
    
    
    # ------------------- 2. 合并其他交易特征（按 from 聚合） -------------------
    # 排除的列
    exclude_cols = ['from', 'to', 'position']
    # 保留需要合并的列
    extra_cols = [col for col in df.columns if col not in exclude_cols]
    
    if extra_cols:
        # 按 'from' 聚合
        agg_dict = {}
        for col in extra_cols:
            if pd.api.types.is_numeric_dtype(df[col]):
                agg_dict[col] = 'sum'          # 数值列求和（如金额）
            else:
                agg_dict[col] = 'first'        # 非数值列取第一个（可改为 'nunique' / 'count' 等）
        
        # 聚合
        extra_features = df.groupby('from')[extra_cols].agg(agg_dict)
        
        # 添加前缀避免列名冲突（如 amount_sum）
        extra_features = extra_features.add_prefix('from_')
        
        # 合并到 features_df（左连接，缺失补 0 或 NaN）
        features_df = features_df.join(extra_features, how='left')
        
        # 可选：数值列缺失值填 0
        numeric_cols = extra_features.select_dtypes(include='number').columns
        features_df[numeric_cols] = features_df[numeric_cols].fillna(0)
    
    # ------------------- 3. 类型转换 -------------------
    # 确保出度/入度等为整数
    int_cols = ['out_degree', 'in_degree', 'total_tx', 'net_flow']
    features_df[int_cols] = features_df[int_cols].astype(int)
    
    return features_df

In [7]:
# 目标字典，现在用于存储包含 trace_graph_df, node_features 和 label 的字典
ponzi_contract_data = {} 

# 1. 确保获取的是唯一的地址列表
unique_addresses = list(set(ponzi_contract['address']))

for addr in unique_addresses:
    # 2.1 构造 Trace Graph DataFrame
    # 假设 construct_EthTrace_graph(ponzi_contract, addr) 返回一个 DataFrame
    TraceChain_df, hyperedge_dict = construct_EthTrace_graph(ponzi_contract, addr, opcode_categories)
    transactional_features = extract_transactional_features(TraceChain_df)
    
    # 2.3 合并节点特征
    # 账户ID作为索引。fillna(0) 处理拓扑特征中可能因图太小或隔离节点导致的 NaN
    node_features = transactional_features
    
    # 2.4 获取合约标签
    contract_label = label_value = ponzi_contract[ponzi_contract['address'] == addr]['label'].iloc[0]
    
    ponzi_contract_data[addr] = {
        'trace_graph_df': TraceChain_df,
        'node_features': node_features,
        'hyperedge_dict': hyperedge_dict,
        'label': contract_label
    }

In [8]:
def align_node_features(ponzi_contract_data):
    """
    统一所有 node_features 的列（列对齐 + 补0）
    """
    all_feature_dfs = [item['node_features'] for item in ponzi_contract_data.values()]
    
    # 找出所有图中出现过的列
    all_columns = set()
    for df in all_feature_dfs:
        all_columns.update(df.columns)
    all_columns = sorted(list(all_columns))
    
    print(f"统一特征维度: {len(all_columns)} 列 → {all_columns}")
    
    # 对每个图的 node_features 做 reindex + fillna(0)
    for addr, item in ponzi_contract_data.items():
        df = item['node_features']
        df_aligned = df.reindex(columns=all_columns, fill_value=0)
        item['node_features'] = df_aligned.astype('float32')  # 确保类型一致
    
    return ponzi_contract_data, all_columns

In [9]:
ponzi_contract_data, all_columns = align_node_features(ponzi_contract_data)

统一特征维度: 146 列 → ['from_opcode_ADD', 'from_opcode_ADDMOD', 'from_opcode_ADDRESS', 'from_opcode_AND', 'from_opcode_BALANCE', 'from_opcode_BLOCKHASH', 'from_opcode_BYTE', 'from_opcode_CALL', 'from_opcode_CALLCODE', 'from_opcode_CALLDATACOPY', 'from_opcode_CALLDATALOAD', 'from_opcode_CALLDATASIZE', 'from_opcode_CALLER', 'from_opcode_CALLVALUE', 'from_opcode_CHAINID', 'from_opcode_CODECOPY', 'from_opcode_CODESIZE', 'from_opcode_COINBASE', 'from_opcode_CREATE', 'from_opcode_CREATE2', 'from_opcode_DELEGATECALL', 'from_opcode_DIFFICULTY', 'from_opcode_DIV', 'from_opcode_DUP1', 'from_opcode_DUP10', 'from_opcode_DUP11', 'from_opcode_DUP12', 'from_opcode_DUP13', 'from_opcode_DUP14', 'from_opcode_DUP15', 'from_opcode_DUP16', 'from_opcode_DUP2', 'from_opcode_DUP3', 'from_opcode_DUP4', 'from_opcode_DUP5', 'from_opcode_DUP6', 'from_opcode_DUP7', 'from_opcode_DUP8', 'from_opcode_DUP9', 'from_opcode_EQ', 'from_opcode_EXP', 'from_opcode_EXTCODECOPY', 'from_opcode_EXTCODEHASH', 'from_opcode_EXTCODESIZE',

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from scipy.sparse import coo_matrix
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
import random
from tqdm import tqdm
import os

# -----------------------------------------------------------
# 1. 超参数配置
# -----------------------------------------------------------
MIN_CLUSTER_SIZE = 2
RANDOM_SEED = 2025
EPOCHS = 100
LEARNING_RATE = 0.0001
BATCH_SIZE = 32
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 设置随机种子
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print(f"Using device: {DEVICE}")

# -----------------------------------------------------------
# 2. Focal Loss 定义（已修复 alpha 设备问题）
# -----------------------------------------------------------

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.75, reduction='sum'):
        super(FocalLoss, self).__init__()
        # gamma (float): 聚焦参数，用于调整难易样本的损失贡献
        self.gamma = gamma
        # alpha (float/tensor): 类别平衡参数，用于调整正负样本的权重
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, input, target):
        # input: [batch_size, num_classes] (Logits)
        # target: [batch_size] (Class indices)
        
        # 1. 计算标准交叉熵损失
        # reduction='none' 确保返回每个样本的损失值，以便后续计算 (1-p_t)^gamma
        ce_loss = F.cross_entropy(input, target, reduction='none')
        
        # 2. 计算 p_t: 模型对真实类别的预测概率
        # p_t = exp(-ce_loss)
        p_t = torch.exp(-ce_loss)
        
        # 3. 计算 modulating factor: (1 - p_t)^gamma
        focal_term = (1.0 - p_t)**self.gamma
        
        # 4. 计算 alpha_t
        # 注意: 您的任务是二分类 (Ponzi/Non-Ponzi)，假设 Ponzi 是类别 1。
        # target_one_hot = F.one_hot(target, num_classes=input.size(-1)).float()
        # if input.size(-1) == 2:
        #     # 简单二分类 alpha_t: 类别 0 权重 (1-alpha)，类别 1 权重 (alpha)
        #     alpha_t = self.alpha * target_one_hot[:, 1] + (1 - self.alpha) * target_one_hot[:, 0]
        # else:
        #     # 对于多分类或更通用的情况，使用类别索引查找
        #     # 简单的 alpha 实现: 假设 alpha 是一个包含 num_classes 权重的 Tensor
        #     # 如果 alpha 是一个 float，则默认只用于 minority class (index=1)
        #     alpha_t = torch.ones_like(target).float()
        #     alpha_t[target == 1] = self.alpha 
        #     alpha_t[target == 0] = 1 - self.alpha
        
        # 简化版 alpha_t (假设 alpha 是应用于少数类 1 的权重)
        if isinstance(self.alpha, (float, int)):
            alpha_t = torch.ones_like(target).float() * (1 - self.alpha)
            # 假设少数类是 1 (Ponzi)
            alpha_t[target == 1] = self.alpha 
        else:
             # 如果 alpha 是一个权重张量，则可以直接使用 target 索引
             alpha_t = self.alpha[target]

        # 5. 计算 Focal Loss: - alpha_t * (1 - p_t)^gamma * log(p_t)
        # 由于 -log(p_t) = ce_loss，所以 Loss = alpha_t * focal_term * ce_loss
        loss = alpha_t * focal_term * ce_loss
        
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss

# -----------------------------------------------------------
# 3. 模型定义 (HypergraphConv, HGCNN) - 未修改
# -----------------------------------------------------------
class HypergraphConv(nn.Module):
    def __init__(self, in_features, out_features, dropout=0.5):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        nn.init.xavier_uniform_(self.linear.weight)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X, H, D_v_inv, D_e_inv):
        HX = torch.sparse.mm(H.t(), X)
        D_e_inv_HX = torch.mm(D_e_inv, HX)
        Ht_D_e_inv_HX = torch.sparse.mm(H, D_e_inv_HX)
        X_out = torch.mm(D_v_inv, Ht_D_e_inv_HX)
        X_out = self.linear(X_out)
        X_out = F.relu(X_out)
        return self.dropout(X_out)


class HGCNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers, dropout, num_classes=2):
        super().__init__()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()

        self.convs.append(HypergraphConv(input_dim, hidden_dim, dropout))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 2):
            self.convs.append(HypergraphConv(hidden_dim, hidden_dim, dropout))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.convs.append(HypergraphConv(hidden_dim, output_dim, dropout))
        self.bns.append(nn.BatchNorm1d(output_dim))

        self.classifier = nn.Sequential(
            nn.Linear(output_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )
        self.reset_parameters()

    def reset_parameters(self):
        for conv in self.convs:
            nn.init.xavier_uniform_(conv.linear.weight)
        for bn in self.bns:
            bn.reset_parameters()
        for layer in self.classifier:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)

    def forward(self, X, H, D_v_inv, D_e_inv, batch=None):
        h = X
        for conv, bn in zip(self.convs, self.bns):
            h = conv(h, H, D_v_inv, D_e_inv)
            h = bn(h)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)

        # 图级池化
        if batch is None:
            graph_emb = torch.mean(h, dim=0, keepdim=True)
        else:
            # 确保 batch 索引是正确的
            num_graphs = batch.max().item() + 1
            graph_emb_list = []
            for i in range(num_graphs):
                mask = (batch == i)
                if mask.sum() > 0:
                    graph_emb_list.append(h[mask].mean(dim=0))
            graph_emb = torch.stack(graph_emb_list, dim=0)

        logits = self.classifier(graph_emb)
        return logits, graph_emb


# -----------------------------------------------------------
# 4. 数据处理函数 - 未修改
# -----------------------------------------------------------
def build_hypergraph_based(ponzi_contract_data, device=DEVICE):
    graphs_data = {}
    for addr, data in ponzi_contract_data.items():
        node_features = data['node_features']
        hyperedge_dict = data['hyperedge_dict']
        label = data['label']

        if isinstance(node_features, dict):
            node_feature_list = list(node_features.values())
        else:
            node_feature_list = node_features

        if len(node_feature_list) == 0:
            # print(f"[Skipped] {addr}: 节点特征为空")
            continue

        # 确保 features 位于 DEVICE
        features = torch.tensor(np.array(node_feature_list), dtype=torch.float32, device=device)
        N = features.shape[0]
        hyperedges = []
        for edge_lists in hyperedge_dict.values():
            for edge_nodes in edge_lists:
                # 假设 node_features 的键是索引 (0, 1, ..., N-1)
                # 验证索引是否在范围内
                valid_nodes = [idx for idx in edge_nodes if 0 <= idx < N]
                if len(valid_nodes) >= MIN_CLUSTER_SIZE:
                    hyperedges.append(valid_nodes)

        M = len(hyperedges)
        if M == 0 or N == 0:
            # print(f"[Skipped] {addr}: N={N}, M={M}")
            continue

        # ========== 构造超图关联矩阵 H ==========
        row, col = [], []
        for e_idx, nodes in enumerate(hyperedges):
            for n_idx in nodes:
                row.append(n_idx)
                col.append(e_idx)

        # 确保 indices, values, H 位于 DEVICE
        indices = torch.tensor([row, col], dtype=torch.long, device=device)
        values = torch.ones(len(row), dtype=torch.float, device=device)
        H = torch.sparse_coo_tensor(indices, values, (N, M)).to(device)

        D_v = torch.zeros(N, device=device)
        D_v.scatter_add_(0, indices[0], torch.ones_like(indices[0], dtype=torch.float))
        D_v_inv = torch.diag(1.0 / (D_v + 1e-8)) # 密集对角矩阵

        D_e = torch.tensor([len(nodes) for nodes in hyperedges], dtype=torch.float, device=device)
        D_e_inv = torch.diag(1.0 / (D_e + 1e-8)) # 密集对角矩阵

        graphs_data[addr] = {
            'features': features,
            'H': H,
            'D_v_inv': D_v_inv,
            'D_e_inv': D_e_inv,
            # 确保 label 位于 DEVICE
            'label': torch.tensor([label], dtype=torch.long, device=device),
            'num_nodes': N,
            'num_hyperedges': M
        }

    final_list = list(graphs_data.values())
    print(f"构建完成: {len(final_list)} 个超图样本")
    return final_list


def split_ponzi_data(ponzi_contract_data, train_ratio=0.525, val_ratio=0.175, test_ratio=0.3, seed=RANDOM_SEED):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6
    addrs = list(ponzi_contract_data.keys())
    labels = [ponzi_contract_data[a]['label'] for a in addrs]

    train_addrs, temp_addrs, _, temp_labels = train_test_split(
        addrs, labels, train_size=train_ratio, random_state=seed, stratify=labels
    )
    val_ratio_adj = val_ratio / (val_ratio + test_ratio)
    val_addrs, test_addrs = train_test_split(
        temp_addrs, train_size=val_ratio_adj, random_state=seed, stratify=temp_labels
    )

    train_data = {a: ponzi_contract_data[a] for a in train_addrs}
    val_data = {a: ponzi_contract_data[a] for a in val_addrs}
    test_data = {a: ponzi_contract_data[a] for a in test_addrs}

    print(f"总样本: {len(addrs)} | 训练: {len(train_data)} | 验证: {len(val_data)} | 测试: {len(test_data)}")
    return train_data, val_data, test_data


# -----------------------------------------------------------
# 5. Dataset & Collate - 未修改
# -----------------------------------------------------------
class HypergraphDataset(Dataset):
    def __init__(self, graphs): self.graphs = graphs
    def __len__(self): return len(self.graphs)
    def __getitem__(self, idx): return self.graphs[idx]

def collate_fn(batch):
    features_list, H_rows, H_cols, H_vals, labels_list = [], [], [], [], []
    node_batch_idx = []
    node_offset = edge_offset = 0

    for i, g in enumerate(batch):
        f = g['features']
        # 确保 H 是稀疏张量且设备正确
        H_i = g['H'].coalesce()
        N_i, M_i = H_i.shape

        features_list.append(f)
        idx = H_i.indices()
        H_rows.append(idx[0] + node_offset)
        H_cols.append(idx[1] + edge_offset)
        H_vals.append(H_i.values())
        labels_list.append(g['label'])
        node_batch_idx.append(torch.full((N_i,), i, dtype=torch.long, device=f.device))

        node_offset += N_i
        edge_offset += M_i

    # X, H_batch, Dv_inv, De_inv, labels, batch_idx 都会被移到 DEVICE (或已经在 DEVICE)
    X = torch.cat(features_list, dim=0)
    batch_idx = torch.cat(node_batch_idx, dim=0)
    labels = torch.cat(labels_list, dim=0).to(DEVICE) # 确保 labels 在 DEVICE

    H_idx = torch.stack([torch.cat(H_rows), torch.cat(H_cols)])
    H_val = torch.cat(H_vals)
    H_batch = torch.sparse_coo_tensor(H_idx, H_val, (node_offset, edge_offset)).to(X.device)

    D_v_inv = torch.diag(torch.cat([g['D_v_inv'].diag() for g in batch]))
    D_e_inv = torch.diag(torch.cat([g['D_e_inv'].diag() for g in batch]))

    return X, H_batch, D_v_inv, D_e_inv, labels, batch_idx


# -----------------------------------------------------------
# 6. 训练与评估 - 未修改
# -----------------------------------------------------------
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for X, H, Dv_inv, De_inv, y, batch in tqdm(loader, desc="Train"):
        optimizer.zero_grad()
        # X, H, Dv_inv, De_inv, y, batch 都在 DEVICE
        logits, _ = model(X, H, Dv_inv, De_inv, batch)
        loss = criterion(logits, y) # FocalLoss 现在可以正确处理不同设备上的 alpha 和 y
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * y.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_y, all_pred, all_prob = [], [], []

    for X, H, Dv_inv, De_inv, y, batch in loader:
        logits, _ = model(X, H, Dv_inv, De_inv, batch)
        loss = criterion(logits, y)
        total_loss += loss.item() * y.size(0)

        prob = F.softmax(logits, dim=1)[:, 1]
        pred = logits.argmax(1)

        all_y.extend(y.cpu().tolist())
        all_pred.extend(pred.cpu().tolist())
        all_prob.extend(prob.cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, all_y, all_pred, all_prob
# -----------------------------------------------------------

# === 开始实验 ===
train_dict, val_dict, test_dict = split_ponzi_data(ponzi_contract_data, 0.525, 0.175, 0.3, RANDOM_SEED)

train_graphs = build_hypergraph_based(train_dict, DEVICE)
val_graphs = build_hypergraph_based(val_dict, DEVICE)
test_graphs = build_hypergraph_based(test_dict, DEVICE)

# --- 3. 初始化模型 ---
# 检查 input_dim 是否有效
if not train_graphs:
    raise ValueError("训练集构建失败，请检查数据和 MIN_CLUSTER_SIZE 设置！")
input_dim = train_graphs[0]['features'].shape[1]

# 创建 DataLoader
train_loader = DataLoader(HypergraphDataset(train_graphs), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(HypergraphDataset(val_graphs), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(HypergraphDataset(test_graphs), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# 模型参数
model = HGCNN(input_dim=input_dim, hidden_dim=256, output_dim=8, num_layers=3, dropout=0.5, num_classes=2).to(DEVICE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=5e-4)
# optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
# # 替换 nn.CrossEntropyLoss() 为 FocalLoss
# # 建议的超参数：
# # gamma=2.0 (增加对难分样本的关注)
# # alpha=0.75 (增加少数类 Ponzi (假设为 1) 的权重, 0.75 > 0.5)
# criterion = FocalLoss(gamma=2.0, alpha=0.75, reduction='mean') 
# # 或者如果使用 Class-Balanced Weighted Cross-Entropy:
# # class_weights = torch.tensor([weight_majority, weight_minority], dtype=torch.float32).to(device)
# # criterion = nn.CrossEntropyLoss(weight=class_weights)

# === 训练循环 ===
best_val_f1 = 0
print("\n" + "="*60)
print("开始训练 (批处理超图)")
print("="*60)

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_y, val_pred, val_prob = evaluate(model, val_loader, criterion)

    val_acc = accuracy_score(val_y, val_pred)
    val_auc = roc_auc_score(val_y, val_prob)
    # 修复：确保 val_pred 不为空
    if len(val_y) > 0:
        val_f1 = f1_score(val_y, val_pred, average='weighted', zero_division=0)
    else:
        val_f1 = 0.0

    print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f} | Val F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), "best_hgcn.pt")
        print("  → 新最佳模型已保存！")

# === 加载最佳模型并测试 ===
if os.path.exists("best_hgcn.pt"):
    model.load_state_dict(torch.load("best_hgcn.pt", map_location=DEVICE))
    test_loss, test_y, test_pred, test_prob = evaluate(model, test_loader, criterion)

    # 修复：确保 test_pred 不为空
    if len(test_y) > 0:
        test_acc = accuracy_score(test_y, test_pred)
        test_auc = roc_auc_score(test_y, test_prob)
        test_f1 = f1_score(test_y, test_pred, average='weighted', zero_division=0)
        test_precision = precision_score(test_y, test_pred, average='weighted', zero_division=0)
        test_recall = recall_score(test_y, test_pred, average='weighted', zero_division=0)
        cm = confusion_matrix(test_y, test_pred)
    else:
        # 如果测试集为空，则设置默认值
        test_acc = test_auc = test_f1 = test_precision = test_recall = 0.0
        cm = np.array([[0, 0], [0, 0]])


    print("\n" + "="*60)
    print("测试集最终性能 (1=异常/Ponzi, 0=正常)")
    print("="*60)
    print(f"Loss     : {test_loss:.4f}")
    print(f"Accuracy : {test_acc:.4f}")
    print(f"AUC      : {test_auc:.4f}")
    print(f"F1       : {test_f1:.4f}")
    print(f"Precision: {test_precision:.4f}")
    print(f"Recall   : {test_recall:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    print("="*60)
else:
    print("\n" + "="*60)
    print("未找到保存的最佳模型，跳过测试步骤。")
    print("="*60)

Using device: cuda
总样本: 6468 | 训练: 3395 | 验证: 1132 | 测试: 1941
构建完成: 3395 个超图样本
构建完成: 1132 个超图样本
构建完成: 1941 个超图样本

开始训练 (批处理超图)


Train: 100%|██████████| 107/107 [00:03<00:00, 35.11it/s]


Epoch 001 | Train Loss: 0.5881 | Val Loss: 0.4880 | Val Acc: 0.9541 | Val AUC: 0.7163 | Val F1: 0.9316
  → 新最佳模型已保存！


Train: 100%|██████████| 107/107 [00:02<00:00, 36.49it/s]


Epoch 002 | Train Loss: 0.4355 | Val Loss: 0.3859 | Val Acc: 0.9541 | Val AUC: 0.7913 | Val F1: 0.9316


...

Epoch 098 | Train Loss: 0.0441 | Val Loss: 0.1400 | Val Acc: 0.9664 | Val AUC: 0.9721 | Val F1: 0.9692


Train: 100%|██████████| 107/107 [00:03<00:00, 34.56it/s]


Epoch 099 | Train Loss: 0.0398 | Val Loss: 0.1405 | Val Acc: 0.9558 | Val AUC: 0.9700 | Val F1: 0.9611


Train: 100%|██████████| 107/107 [00:03<00:00, 34.18it/s]


Epoch 100 | Train Loss: 0.0453 | Val Loss: 0.1783 | Val Acc: 0.9408 | Val AUC: 0.9753 | Val F1: 0.9507

测试集最终性能 (1=异常/Ponzi, 0=正常)
Loss     : 0.1161
Accuracy : 0.9758
AUC      : 0.9524
F1       : 0.9761
Precision: 0.9765
Recall   : 0.9758
Confusion Matrix:
[[1825   26]
 [  21   69]]


In [20]:
# === 加载最佳模型并测试 ===
if os.path.exists("best_hgcn.pt"):
    model.load_state_dict(torch.load("best_hgcn.pt", map_location=DEVICE))
    test_loss, test_y, test_pred, test_prob = evaluate(model, test_loader, criterion)

    # 修复：确保 test_pred 不为空
    if len(test_y) > 0:
        test_acc = accuracy_score(test_y, test_pred)
        test_auc = roc_auc_score(test_y, test_prob)
        test_f1 = f1_score(test_y, test_pred, zero_division=0)
        test_precision = precision_score(test_y, test_pred, zero_division=0)
        test_recall = recall_score(test_y, test_pred, zero_division=0)
        cm = confusion_matrix(test_y, test_pred)
    else:
        # 如果测试集为空，则设置默认值
        test_acc = test_auc = test_f1 = test_precision = test_recall = 0.0
        cm = np.array([[0, 0], [0, 0]])


    print("\n" + "="*60)
    print("测试集最终性能 (1=异常/Ponzi, 0=正常)")
    print("="*60)
    print(f"Loss     : {test_loss:.4f}")
    print(f"Accuracy : {test_acc:.4f}")
    print(f"AUC      : {test_auc:.4f}")
    print(f"F1       : {test_f1:.4f}")
    print(f"Precision: {test_precision:.4f}")
    print(f"Recall   : {test_recall:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    print("="*60)
else:
    print("\n" + "="*60)
    print("未找到保存的最佳模型，跳过测试步骤。")
    print("="*60)


测试集最终性能 (1=异常/Ponzi, 0=正常)
Loss     : 0.1161
Accuracy : 0.9758
AUC      : 0.9524
F1       : 0.7459
Precision: 0.7263
Recall   : 0.7667
Confusion Matrix:
[[1825   26]
 [  21   69]]
